## Part A - Scikit-learn Implementation

### 1. Import libraries and load the dataset

In [4]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Load dataset
df = pd.read_csv("data/garments_worker_productivity.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


In [6]:
# ============================================
# PREPARE DATA
# ============================================

# Remove date because it is not being used as a feature
df_model = df.drop(columns=["date"]).copy()

# Create classification target
# 1 = actual productivity meets/exceeds target
# 0 = actual productivity is below target

df_model["MeetsTarget"] = (
    df_model["actual_productivity"]
    >= df_model["targeted_productivity"]
).astype(int)

print("Dataset shape:", df_model.shape)

print("\nMeetsTarget distribution:")
print(df_model["MeetsTarget"].value_counts())

print("\nMissing values:")
print(df_model.isnull().sum())

Dataset shape: (1197, 15)

MeetsTarget distribution:
MeetsTarget
1    875
0    322
Name: count, dtype: int64

Missing values:
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
MeetsTarget                0
dtype: int64


In [7]:
# ============================================
# SEPARATE FEATURES AND TARGETS
# ============================================

# Regression
X_reg = df_model.drop(
    columns=["actual_productivity", "MeetsTarget"]
)

y_reg = df_model["actual_productivity"]


# Classification
# actual_productivity MUST NOT be used as an input
X_cls = df_model.drop(
    columns=["actual_productivity", "MeetsTarget"]
)

y_cls = df_model["MeetsTarget"]


# ============================================
# FIXED TRAIN-TEST SPLIT
# ============================================

train_idx, test_idx = train_test_split(
    np.arange(len(df_model)),
    test_size=0.20,
    random_state=42,
    stratify=y_cls
)


# Save the exact indices
np.save("train_indices.npy", train_idx)
np.save("test_indices.npy", test_idx)


# ============================================
# CREATE TRAINING AND TESTING DATA
# ============================================

# Regression
X_reg_train = X_reg.iloc[train_idx]
X_reg_test = X_reg.iloc[test_idx]

y_reg_train = y_reg.iloc[train_idx]
y_reg_test = y_reg.iloc[test_idx]


# Classification
X_cls_train = X_cls.iloc[train_idx]
X_cls_test = X_cls.iloc[test_idx]

y_cls_train = y_cls.iloc[train_idx]
y_cls_test = y_cls.iloc[test_idx]


# ============================================
# CHECK
# ============================================

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))

print("\nRegression:")
print("X_train:", X_reg_train.shape)
print("X_test :", X_reg_test.shape)

print("\nClassification:")
print("X_train:", X_cls_train.shape)
print("X_test :", X_cls_test.shape)

Training samples: 957
Testing samples: 240

Regression:
X_train: (957, 13)
X_test : (240, 13)

Classification:
X_train: (957, 13)
X_test : (240, 13)


### 2. Understand the dataset

In [3]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nSummary statistics:")
df.describe(include="all")

Column names:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null  

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
count,1197,1197,1197,1197,1197.000000,1197.000000,1197.000000,691.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000,1197.000000
unique,59,5,3,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,1/31/2015,Quarter1,sweing,Wednesday,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,24,360,691,208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,6.426901,0.729632,15.062172,1190.465991,4567.460317,38.210526,0.730159,0.369256,0.150376,34.609858,0.735091
std,NaN,NaN,NaN,NaN,3.463963,0.097891,10.943219,1837.455001,3348.823563,160.182643,12.709757,3.268987,0.427848,22.197687,0.174488
min,NaN,NaN,NaN,NaN,1.000000,0.070000,2.900000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.233705
25%,NaN,NaN,NaN,NaN,3.000000,0.700000,3.940000,774.500000,1440.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.650307
50%,NaN,NaN,NaN,NaN,6.000000,0.750000,15.260000,1039.000000,3960.000000,0.000000,0.000000,0.000000,0.000000,34.000000,0.773333
75%,NaN,NaN,NaN,NaN,9.000000,0.800000,24.260000,1252.500000,6960.000000,50.000000,0.000000,0.000000,0.000000,57.000000,0.850253


### 3. Basic preprocessing

In [4]:
# Remove date column because it is not directly used as a numerical/categorical predictor
df_model = df.drop(columns=["date"])

# Create classification target
df_model["MeetsTarget"] = (
    df_model["actual_productivity"] >= df_model["targeted_productivity"]
).astype(int)

print(df_model.head())
print("\nMeetsTarget counts:")
print(df_model["MeetsTarget"].value_counts())

    quarter  department       day  team  targeted_productivity    smv     wip  \
0  Quarter1      sweing  Thursday     8                   0.80  26.16  1108.0   
1  Quarter1  finishing   Thursday     1                   0.75   3.94     NaN   
2  Quarter1      sweing  Thursday    11                   0.80  11.41   968.0   
3  Quarter1      sweing  Thursday    12                   0.80  11.41   968.0   
4  Quarter1      sweing  Thursday     6                   0.80  25.90  1170.0   

   over_time  incentive  idle_time  idle_men  no_of_style_change  \
0       7080         98        0.0         0                   0   
1        960          0        0.0         0                   0   
2       3660         50        0.0         0                   0   
3       3660         50        0.0         0                   0   
4       1920         50        0.0         0                   0   

   no_of_workers  actual_productivity  MeetsTarget  
0           59.0             0.940725            1 

### 4. Create the common train-test split

In [8]:
# ============================================
# SEPARATE FEATURES AND TARGETS
# ============================================

# Regression
X_reg = df_model.drop(
    columns=["actual_productivity", "MeetsTarget"]
)

y_reg = df_model["actual_productivity"]


# Classification
# actual_productivity MUST NOT be used as an input
X_cls = df_model.drop(
    columns=["actual_productivity", "MeetsTarget"]
)

y_cls = df_model["MeetsTarget"]


# ============================================
# FIXED TRAIN-TEST SPLIT FOR ALL MODELS
# ============================================

# Use the same train-test samples for:
# 1. Scikit-learn models
# 2. From-scratch models

train_idx, test_idx = train_test_split(
    np.arange(len(df_model)),
    test_size=0.20,
    random_state=42,
    stratify=y_cls
)


# Save the exact indices so the from-scratch
# notebook can use the same samples
np.save("train_indices.npy", train_idx)
np.save("test_indices.npy", test_idx)


# Create regression train/test data
X_reg_train = X_reg.iloc[train_idx]
X_reg_test = X_reg.iloc[test_idx]

y_reg_train = y_reg.iloc[train_idx]
y_reg_test = y_reg.iloc[test_idx]


# Create classification train/test data
X_cls_train = X_cls.iloc[train_idx]
X_cls_test = X_cls.iloc[test_idx]

y_cls_train = y_cls.iloc[train_idx]
y_cls_test = y_cls.iloc[test_idx]


# Display split information
print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))

print("\nRegression training shape:", X_reg_train.shape)
print("Regression testing shape:", X_reg_test.shape)

print("\nClassification training shape:", X_cls_train.shape)
print("Classification testing shape:", X_cls_test.shape)

Training samples: 957
Testing samples: 240

Regression training shape: (957, 13)
Regression testing shape: (240, 13)

Classification training shape: (957, 13)
Classification testing shape: (240, 13)


### 5. Identify numerical and categorical columns

In [6]:
numeric_features = X_reg.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
categorical_features = X_reg.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']

Categorical features:
['quarter', 'department', 'day']


C:\Users\George Mathew\AppData\Local\Temp\ipykernel_13864\4020088797.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_reg.select_dtypes(include=["object"]).columns.tolist()


### 6. Create the preprocessing pipeline

In [7]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

### 7. Linear Regression

In [10]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

# Training time
start_time = time.perf_counter()

linear_model.fit(X_reg_train, y_reg_train)

linear_training_time = time.perf_counter() - start_time

# Prediction time
start_time = time.perf_counter()

y_reg_pred = linear_model.predict(X_reg_test)

linear_prediction_time = time.perf_counter() - start_time

# Evaluation
mae = mean_absolute_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2 = r2_score(y_reg_test, y_reg_pred)

print("Linear Regression Results")
print("-------------------------")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("Training time:", linear_training_time, "seconds")
print("Prediction time:", linear_prediction_time, "seconds")

Linear Regression Results
-------------------------
MAE: 0.10727843302917223
RMSE: 0.1437402979105884
R2: 0.2885214314434412
Training time: 0.9064114000066184 seconds
Prediction time: 0.043647499987855554 seconds


### 8. Logistic Regression

In [11]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000))
    ]
)

# Training time
start_time = time.perf_counter()

logistic_model.fit(X_cls_train, y_cls_train)

logistic_training_time = time.perf_counter() - start_time

# Prediction time
start_time = time.perf_counter()

y_cls_pred = logistic_model.predict(X_cls_test)

logistic_prediction_time = time.perf_counter() - start_time

# Evaluation
accuracy = accuracy_score(y_cls_test, y_cls_pred)
precision = precision_score(y_cls_test, y_cls_pred, zero_division=0)
recall = recall_score(y_cls_test, y_cls_pred, zero_division=0)
f1 = f1_score(y_cls_test, y_cls_pred, zero_division=0)

print("Logistic Regression Results")
print("---------------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Training time:", logistic_training_time, "seconds")
print("Prediction time:", logistic_prediction_time, "seconds")

Logistic Regression Results
---------------------------
Accuracy: 0.7083333333333334
Precision: 0.7611940298507462
Recall: 0.8742857142857143
F1-score: 0.8138297872340425
Training time: 0.3179674999846611 seconds
Prediction time: 0.0509643999976106 seconds


### 9. Make a Part A results table

In [12]:
sklearn_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Logistic Regression"
    ],
    "MAE": [
        mae,
        np.nan
    ],
    "RMSE": [
        rmse,
        np.nan
    ],
    "R2": [
        r2,
        np.nan
    ],
    "Accuracy": [
        np.nan,
        accuracy
    ],
    "Precision": [
        np.nan,
        precision
    ],
    "Recall": [
        np.nan,
        recall
    ],
    "F1": [
        np.nan,
        f1
    ],
    "Training Time (s)": [
        linear_training_time,
        logistic_training_time
    ],
    "Prediction Time (s)": [
        linear_prediction_time,
        logistic_prediction_time
    ]
})

sklearn_results

,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,0.906411,0.043647
1,Logistic Regression,NaN,NaN,NaN,0.708333,0.761194,0.874286,0.81383,0.317967,0.050964


In [15]:
# ============================================
# RE-RUN SKLEARN MODELS AND SAVE RESULTS
# ============================================

# --------------------------------------------
# 1. PREPROCESSING
# --------------------------------------------

numeric_features = X_reg.select_dtypes(
    include=["int64", "float64", "bool"]
).columns.tolist()

categorical_features = X_reg.select_dtypes(
    include=["object"]
).columns.tolist()


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# --------------------------------------------
# 2. LINEAR REGRESSION
# --------------------------------------------

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

# Training
start_time = time.perf_counter()

linear_model.fit(
    X_reg_train,
    y_reg_train
)

linear_training_time = time.perf_counter() - start_time


# Prediction
start_time = time.perf_counter()

y_reg_pred = linear_model.predict(
    X_reg_test
)

linear_prediction_time = time.perf_counter() - start_time


# Metrics
mae = mean_absolute_error(
    y_reg_test,
    y_reg_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_reg_test,
        y_reg_pred
    )
)

r2 = r2_score(
    y_reg_test,
    y_reg_pred
)


# --------------------------------------------
# 3. LOGISTIC REGRESSION
# --------------------------------------------

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000))
    ]
)

# Training
start_time = time.perf_counter()

logistic_model.fit(
    X_cls_train,
    y_cls_train
)

logistic_training_time = time.perf_counter() - start_time


# Prediction
start_time = time.perf_counter()

y_cls_pred = logistic_model.predict(
    X_cls_test
)

logistic_prediction_time = time.perf_counter() - start_time


# Metrics
accuracy = accuracy_score(
    y_cls_test,
    y_cls_pred
)

precision = precision_score(
    y_cls_test,
    y_cls_pred,
    zero_division=0
)

recall = recall_score(
    y_cls_test,
    y_cls_pred,
    zero_division=0
)

f1 = f1_score(
    y_cls_test,
    y_cls_pred,
    zero_division=0
)


# --------------------------------------------
# 4. CREATE RESULTS TABLE
# --------------------------------------------

sklearn_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Logistic Regression"
    ],

    "MAE": [
        mae,
        np.nan
    ],

    "RMSE": [
        rmse,
        np.nan
    ],

    "R2": [
        r2,
        np.nan
    ],

    "Accuracy": [
        np.nan,
        accuracy
    ],

    "Precision": [
        np.nan,
        precision
    ],

    "Recall": [
        np.nan,
        recall
    ],

    "F1": [
        np.nan,
        f1
    ],

    "Training Time (s)": [
        linear_training_time,
        logistic_training_time
    ],

    "Prediction Time (s)": [
        linear_prediction_time,
        logistic_prediction_time
    ]
})


# Display results
print("SCIKIT-LEARN RESULTS")
print("====================")

display(sklearn_results)


# --------------------------------------------
# 5. SAVE RESULTS
# --------------------------------------------

sklearn_results.to_csv(
    "sklearn_results.csv",
    index=False
)

print("\nsklearn_results.csv created successfully!")

C:\Users\George Mathew\AppData\Local\Temp\ipykernel_10152\4110672417.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_reg.select_dtypes(


SCIKIT-LEARN RESULTS


,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,2.003022,0.051057
1,Logistic Regression,NaN,NaN,NaN,0.708333,0.761194,0.874286,0.81383,0.345022,0.046369



sklearn_results.csv created successfully!


In [16]:
# ============================================
# CREATE AND SAVE SKLEARN RESULTS
# ============================================

sklearn_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Logistic Regression"
    ],

    "MAE": [
        mae,
        np.nan
    ],

    "RMSE": [
        rmse,
        np.nan
    ],

    "R2": [
        r2,
        np.nan
    ],

    "Accuracy": [
        np.nan,
        accuracy
    ],

    "Precision": [
        np.nan,
        precision
    ],

    "Recall": [
        np.nan,
        recall
    ],

    "F1": [
        np.nan,
        f1
    ],

    "Training Time (s)": [
        linear_training_time,
        logistic_training_time
    ],

    "Prediction Time (s)": [
        linear_prediction_time,
        logistic_prediction_time
    ]
})

display(sklearn_results)

# Create the CSV file
sklearn_results.to_csv(
    "sklearn_results.csv",
    index=False
)

print("sklearn_results.csv created successfully!")

,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,2.003022,0.051057
1,Logistic Regression,NaN,NaN,NaN,0.708333,0.761194,0.874286,0.81383,0.345022,0.046369


sklearn_results.csv created successfully!
